# Cogeneration: combined historical data EDA

Explore every file in `data/raw/cogeneration` together, following `1_co2_data_eda.ipynb`, `2_sun_eda.ipynb`, and the wind EDA notebooks. Check schema, hourly coverage, missing values, duplicates, plausibility, distribution, calendar patterns, persistence, low-generation spells, and association with the CO₂ target and renewable generation.

Preserve raw observations. Use UTC for interval alignment and Europe/Amsterdam for calendar profiles. Treat this `WKK Total` series as cogeneration (combined heat and power), not automatically as renewable generation: these exports do not identify fuel composition. Explore it as a possible additional input. Actual generation is the agreed basis because archived forecasts are unavailable; evaluation with future actual inputs is conditional on known production and excludes generation forecast error. No model is fitted here.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs/initial_assignment.md").exists())
DATA_DIR = ROOT / "data/raw/cogeneration"
TARGET = "volume (kWh)"

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.25})

## 1. Load every source file

The directory currently contains CSV exports. Fail visibly if an unexpected file format or schema appears, rather than silently omitting it. Keep source filenames to inspect boundaries and provenance. `validfrom` is the interval start and `validto` is its end; both headers explicitly specify UTC.

In [ ]:
files = sorted(p for p in DATA_DIR.iterdir() if p.is_file())
assert files, "No input files found"
assert all(p.suffix.lower() == ".csv" for p in files), "Review unsupported source files"

frames = [pd.read_csv(p) for p in files]
assert all(frame.columns.equals(frames[0].columns) for frame in frames), "Source schemas differ"

inventory = pd.DataFrame({"file": [p.name for p in files], "bytes": [p.stat().st_size for p in files], "rows": [len(f) for f in frames]})
df = pd.concat([f.assign(source_file=p.name) for p, f in zip(files, frames)], ignore_index=True)
raw_columns = list(frames[0].columns)

for col in ["validfrom (UTC)", "validto (UTC)"]:
    df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
df = df.sort_values("validfrom (UTC)").reset_index(drop=True)

display(inventory)

In [ ]:
display(df.head())

In [ ]:
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "unique_values": df.nunique(dropna=False)}))

In [ ]:
print(f"Total: {len(df):,} rows, {len(raw_columns)} original columns")

`volume (kWh)` is energy generated during each interval and is the feature explored here. Charts convert it to GWh. `capacity (kW)` is a power-valued field: numerical agreement with one-hour energy does not establish installed capacity. The denominator of `percentage` is undocumented; do not interpret it as cogeneration's share of the electricity mix.

The cogeneration export's emissions and emission-factor fields are source-specific and do not replace the total-mix CO₂ target. Inspect their consistency and variation before considering them as features. `Current` does not establish forecast issue times, publication latency, or revision history.

In [ ]:
metadata_columns = ["point", "type", "granularity", "timezone", "activity", "classification"]
display(pd.DataFrame({c: [df[c].unique().tolist()] for c in metadata_columns}).T.rename(columns={0: "observed_values"}))

In [ ]:
numeric_columns = df[raw_columns].select_dtypes(include="number").columns.tolist()
display(df[numeric_columns].describe(percentiles=[.01, .25, .5, .75, .99]).T)

In [ ]:
display(df.groupby("source_file").agg(rows=(TARGET, "size"), start_utc=("validfrom (UTC)", "min"), end_utc_exclusive=("validto (UTC)", "max"), cogeneration_mean_kwh=(TARGET, "mean")))

## 2. Missing values, duplicate rows, and missing hours

A file can have no null cells while still omitting entire hours. Check both, including cross-file boundaries. Do not resolve duplicate timestamps silently. Coverage is measured only between the first and last observed timestamps; this cannot establish completeness outside that range.

In [ ]:
display(pd.DataFrame({"missing": df.isna().sum(), "missing_pct": df.isna().mean() * 100}))

In [ ]:
start = df["validfrom (UTC)"]
end = df["validto (UTC)"]
duplicate_times = start.duplicated(keep=False)
expected = pd.date_range(start.min(), start.max(), freq="h")
missing_hours = expected.difference(pd.DatetimeIndex(start.dropna()))
durations = (end - start).dt.total_seconds() / 3600
checks = pd.Series({
    "duplicate_raw_rows": df.duplicated(raw_columns).sum(),
    "rows_with_duplicate_start": duplicate_times.sum(),
    "invalid_start_or_end": (start.isna() | end.isna()).sum(),
    "non_hour_intervals": durations.ne(1).sum(),
    "off_hour_starts": start.ne(start.dt.floor("h")).sum(),
    "missing_hours_inside_coverage": len(missing_hours),
    "non_contiguous_adjacent_intervals": (start.iloc[1:].reset_index(drop=True) != end.iloc[:-1].reset_index(drop=True)).sum(),
})
display(checks.to_frame("count"))

In [ ]:
missing_table = pd.DataFrame({"missing_hour_utc": missing_hours})
if not missing_table.empty:
    missing_table["run"] = missing_table["missing_hour_utc"].diff().ne(pd.Timedelta(hours=1)).cumsum()
    display(missing_table.groupby("run").agg(
        start_utc=("missing_hour_utc", "min"), end_utc_inclusive=("missing_hour_utc", "max"), hours=("missing_hour_utc", "size"),
    ))
else:
    display(missing_table)

In [ ]:
display(df.loc[duplicate_times].head(10))

In [ ]:
assert not duplicate_times.any() and start.notna().all(), "Resolve timestamp issues before time-series analysis"
s = df.set_index("validfrom (UTC)")[TARGET].reindex(expected)
s.index.name = "time_utc"
local = s.tz_convert("Europe/Amsterdam")
print(f"Local interval coverage: {start.min().tz_convert("Europe/Amsterdam")} to {end.max().tz_convert("Europe/Amsterdam")} (end exclusive)")

## 3. Plausibility and internal consistency

Check non-finite, negative, and zero values and compare energy with power × interval duration. Cogeneration has no obligatory nightly shutdown. Zeros, if present, can reflect operating conditions or reporting issues; these exports cannot distinguish their causes. Retain raw observations. Check whether reported emissions agree with energy × the reported source-specific emission factor, allowing for numerical rounding. This does not validate the underlying emissions accounting or identify fuel composition.

In [ ]:
display(pd.DataFrame({
    "non_finite": (~np.isfinite(df[numeric_columns])).sum(),
    "negative": df[numeric_columns].lt(0).sum(),
    "zero": df[numeric_columns].eq(0).sum(),
}))

In [ ]:
energy_error = df[TARGET] - df["capacity (kW)"] * durations
display(energy_error.describe().to_frame("volume_minus_capacity_times_hours_kwh"))
display(df.loc[energy_error.abs().nlargest(5).index, ["validfrom (UTC)", "validto (UTC)", TARGET, "capacity (kW)"]].assign(
    energy_minus_power_times_hours_kwh=energy_error.loc[energy_error.abs().nlargest(5).index],
))
emission_error = df["emission (kg CO2)"] - df[TARGET] * df["emissionfactor (kg CO2/kWh)"]
display(emission_error.describe().to_frame("reported_minus_energy_times_factor_kg_co2"))
factor = df["emissionfactor (kg CO2/kWh)"]
display(pd.Series({
    "factor_min_kg_per_kwh": factor.min(), "factor_max_kg_per_kwh": factor.max(),
    "factor_range_kg_per_kwh": factor.max() - factor.min(),
    "factor_unique_at_6_decimals": factor.round(6).nunique(),
    "max_absolute_emission_residual_kg": emission_error.abs().max(),
}).to_frame("value"))
print("Inspect tiny factor differences as numerical precision, not evidence of changing fuel mix. Emissions largely repeat energy scaled by the near-constant factor.")

## 4. Distribution and possible outliers

Use 1.5 × IQR fences as descriptive flags, not physical limits. High cogeneration output can be valid. Flag absolute hourly changes above the 99th percentile for review; dispatch, demand, outages, or reporting changes may cause ramps. These files do not identify the causes. Retain every observation.

A zero or near-zero IQR makes flags sensitive to tiny changes. In particular, flags for the almost constant emission factor or mostly zero `percentage` are not evidence of physical anomalies; interpret their magnitude and field definitions first.

In [ ]:
q1, q3 = df[numeric_columns].quantile(.25), df[numeric_columns].quantile(.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
flags = df[numeric_columns].lt(lower) | df[numeric_columns].gt(upper)
display(pd.DataFrame({"lower_fence": lower, "upper_fence": upper, "flagged_rows": flags.sum(), "flagged_pct": flags.mean() * 100}))

In [ ]:
changes = s.diff().abs()
jump_threshold = changes.quantile(.99)
jump_flags = changes.gt(jump_threshold)
display(pd.DataFrame({"cogeneration": s, "absolute_hourly_change": changes}).nlargest(10, "absolute_hourly_change"))

In [ ]:
fig, ax = plt.subplots()
ax.hist(s.dropna() / 1_000_000, bins=60)
ax.set(xlabel="Cogeneration energy (GWh per hourly interval)", ylabel="Hours", title="Combined cogeneration distribution")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.boxplot([g.dropna().to_numpy() / 1_000_000 for _, g in local.groupby(local.index.year)], tick_labels=[str(y) for y in sorted(local.index.year.unique())])
ax.set(xlabel="Local year (2026 partial)", ylabel="Cogeneration energy (GWh per hourly interval)", title="Distribution by year")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.hist(changes.dropna() / 1_000_000, bins=60)
ax.axvline(jump_threshold / 1_000_000, color="red", label="99th percentile")
ax.set(xlabel="Absolute hourly change (GWh)", ylabel="Hours", title="Changes between adjacent hours")
ax.legend()
plt.tight_layout()
plt.show()

### Zero output and low-generation spells

Inspect contiguous UTC zero runs and the zero fraction by local hour. Also inspect periods at or below the full-series 10th percentile. This is an empirical descriptive threshold, not an operational adequacy limit or a feature estimated for model training. Missing hours break runs and are never treated as zero.

In [ ]:
zero = s.eq(0)
zero_runs = zero.ne(zero.shift()).cumsum()
run_table = s.to_frame("cogeneration_kwh").assign(run=zero_runs, zero=zero).groupby("run").agg(
    start_utc=("cogeneration_kwh", lambda x: x.index.min()),
    end_utc_inclusive=("cogeneration_kwh", lambda x: x.index.max()),
    hours=("cogeneration_kwh", "size"), all_zero=("zero", "all"),
)
display(run_table.loc[run_table["all_zero"]].nlargest(10, "hours"))
zero_profile = local.to_frame("cogeneration_kwh").groupby(local.index.hour)["cogeneration_kwh"].agg(
    observations="count", zero_pct=lambda x: x.dropna().eq(0).mean() * 100,
)
display(zero_profile.rename_axis("local_hour"))
low_threshold = s.quantile(.10)
low = s.le(low_threshold) & s.notna()
low_runs = low.ne(low.shift()).cumsum()
low_run_table = s.to_frame("cogeneration_kwh").assign(run=low_runs, low=low).groupby("run").agg(
    start_utc=("cogeneration_kwh", lambda x: x.index.min()),
    end_utc_inclusive=("cogeneration_kwh", lambda x: x.index.max()),
    hours=("cogeneration_kwh", "size"), all_low=("low", "all"),
)
print(f"Low-generation threshold (10th percentile): {low_threshold / 1e6:.3f} GWh per hour")
display(low_run_table.loc[low_run_table["all_low"]].nlargest(10, "hours"))

## 5. Time-series views

Plot hourly observations with daily and monthly means, the latest 28 days, and flagged jumps. Monthly means near dataset boundaries may cover partial months. Monthly coverage counts below make this visible.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(s.index, s / 1_000_000, alpha=.25, linewidth=.4, label="Hourly")
ax.plot(s.resample("D").mean() / 1_000_000, linewidth=.8, label="Daily mean")
ax.plot(s.resample("MS").mean() / 1_000_000, linewidth=2, label="Monthly mean")
ax.scatter(s.index[jump_flags], s[jump_flags] / 1_000_000, s=8, color="red", label="Large hourly change")
ax.set(title="Combined history", ylabel="Cogeneration energy (GWh per hourly interval)", xlabel="UTC")
ax.legend(ncol=4)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
recent = s.loc[s.index.max() - pd.Timedelta(days=28):]
ax.plot(recent.index, recent / 1_000_000)
ax.set(title="Latest 28 days", ylabel="Cogeneration energy (GWh per hourly interval)", xlabel="UTC")
plt.tight_layout()
plt.show()

In [ ]:
display(local.resample("MS").agg(["count", "mean", "min", "max"]).rename_axis("local_month"))

## 6. Daily, weekly, and annual seasonality

Calendar profiles use Dutch local time, including daylight saving transitions. UTC stays regular; local spring days have 23 hours and autumn days 25. Compare years separately because 2026 is incomplete. Counts expose unequal coverage. Profiles show association, not a causal or stationary seasonal effect; fewer than three complete annual cycles limit annual conclusions.

In [ ]:
calendar = local.rename("cogeneration").to_frame()
calendar["year"] = calendar.index.year
calendar["month"] = calendar.index.month
calendar["weekday"] = calendar.index.dayofweek
calendar["hour"] = calendar.index.hour
fig, ax = plt.subplots()

for year, group in calendar.groupby("year"):
    (group.groupby("hour")["cogeneration"].mean() / 1_000_000).plot(ax=ax, marker="o", label=str(year))
ax.set(xlabel="Local hour", ylabel="Mean Cogeneration energy (GWh per hourly interval)", title="Daily profile")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for year, group in calendar.groupby("year"):
    (group.groupby("weekday")["cogeneration"].mean() / 1_000_000).plot(ax=ax, marker="o", label=str(year))
ax.set(xlabel="Local weekday", ylabel="Mean Cogeneration energy (GWh per hourly interval)", title="Weekly profile (Monday = 0)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for year, group in calendar.groupby("year"):
    (group.groupby("month")["cogeneration"].mean() / 1_000_000).plot(ax=ax, marker="o", label=str(year))
ax.set(xlabel="Local month", ylabel="Mean Cogeneration energy (GWh per hourly interval)", title="Annual profile (2026 partial)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
display(calendar.groupby(["year", "month"])["cogeneration"].agg(["count", "mean"]).unstack("year"))

In [ ]:
heatmap = calendar.pivot_table(index="weekday", columns="hour", values="cogeneration", aggfunc="mean") / 1_000_000
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(heatmap, aspect="auto", origin="lower")
ax.set(xticks=range(24), yticks=range(7), yticklabels=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], xlabel="Local hour", title="Mean cogeneration energy by weekday and hour")
fig.colorbar(im, ax=ax, label="GWh")
plt.tight_layout()
plt.show()

### Autocorrelation on a regular hourly grid

Compare raw autocorrelation with autocorrelation after subtracting a centered 30-day rolling mean. The centered smoother is retrospective only and would leak future information as a forecasting feature. Missing hours remain missing. Operational persistence and shared seasonal patterns can produce correlation at daily and weekly lags without a repeating daily or weekly cycle; this plot alone does not establish seasonality or forecast skill.

In [ ]:
detrended = s - s.rolling(24 * 30, center=True, min_periods=24 * 15).mean()
lags = np.arange(1, 24 * 14 + 1)
acf = pd.DataFrame({"raw": [s.autocorr(int(lag)) for lag in lags], "detrended": [detrended.autocorr(int(lag)) for lag in lags]}, index=lags)
ax = acf.plot(figsize=(12, 4))

for lag in [24, 168]:
    ax.axvline(lag, color="grey", linestyle="--", alpha=.7)
ax.set(xlabel="Lag (hours)", ylabel="Pearson autocorrelation", title="Hourly autocorrelation: persistence across hourly lags")
plt.show()

In [ ]:
display(pd.DataFrame({"raw_correlation": [s.autocorr(lag) for lag in [1, 24, 168]], "detrended_correlation": [detrended.autocorr(lag) for lag in [1, 24, 168]], "available_pairs": [(s.notna() & s.shift(lag).notna()).sum() for lag in [1, 24, 168]]}, index=pd.Index([1, 24, 168], name="lag_hours")))

### Cogeneration profile by month and hour

The heatmap describes how mean production varies across local months and hours. Check sample counts and yearly profiles before interpreting seasonal patterns. Monthly energy totals include observed and expected hours so incomplete months remain visible.

In [ ]:
heatmap = calendar.pivot_table(index="month", columns="hour", values="cogeneration", aggfunc="mean") / 1_000_000
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(heatmap, aspect="auto", origin="lower")
ax.set(xticks=range(24), yticks=range(len(heatmap)), yticklabels=heatmap.index, xlabel="Local hour", ylabel="Local month", title="Mean hourly cogeneration energy")
fig.colorbar(im, ax=ax, label="GWh per hourly interval")
plt.tight_layout()
plt.show()
monthly = local.resample("MS").agg(observed_hours="count", energy_kwh=lambda x: x.sum(min_count=1))
monthly["expected_hours"] = [(t + pd.offsets.MonthBegin(1) - t).total_seconds() / 3600 for t in monthly.index]
monthly["coverage_pct"] = monthly["observed_hours"] / monthly["expected_hours"] * 100
monthly["energy_gwh"] = monthly.pop("energy_kwh") / 1_000_000
display(monthly)

## 7. Alignment with the historical CO₂ target

Join on unique UTC interval starts and matching one-hour interval ends. Report unmatched hours in both directions. Correlations are descriptive and may reflect demand, season, and other generation. Cogeneration is part of the electricity mix, so association can also reflect the target's composition; it is not independent evidence of predictive value. Actual generation is the agreed basis for the initial analysis. Its future availability at a forecast origin must be stated: using future actuals in a 168-hour evaluation excludes generation forecast error.

In [ ]:
co2_files = sorted((ROOT / "data/raw/co2").glob("*.csv"))
co2 = pd.concat([pd.read_csv(p) for p in co2_files], ignore_index=True)
for col in ["validfrom (UTC)", "validto (UTC)"]:
    co2[col] = pd.to_datetime(co2[col], utc=True, errors="raise")
assert co2["validfrom (UTC)"].is_unique, "Resolve duplicate CO2 hours before joining"
assert (co2["validto (UTC)"] - co2["validfrom (UTC)"]).eq(pd.Timedelta(hours=1)).all()
co2 = co2.set_index("validfrom (UTC)").sort_index()
cogeneration_end = df.set_index("validfrom (UTC)")["validto (UTC)"]
common = cogeneration_end.index.intersection(co2.index)
assert cogeneration_end.reindex(common).eq(co2.loc[common, "validto (UTC)"]).all()
paired = pd.concat([s.rename("cogeneration_kwh"), co2["emissionfactor (kg CO2/kWh)"].rename("co2_factor")], axis=1).dropna()
display(pd.Series({"paired_hours": len(paired), "cogeneration_hours_without_co2": len(s.dropna().index.difference(co2.index)), "co2_hours_without_cogeneration": len(co2.index.difference(s.dropna().index))}).to_frame("count"))
correlations = pd.Series({"all_hours": paired["cogeneration_kwh"].corr(paired["co2_factor"]), "positive_cogeneration_hours": paired.loc[paired["cogeneration_kwh"].gt(0), "cogeneration_kwh"].corr(paired.loc[paired["cogeneration_kwh"].gt(0), "co2_factor"])})
display(correlations.to_frame("Pearson_correlation"))
display(paired.groupby(paired.index.tz_convert("Europe/Amsterdam").year)[["cogeneration_kwh", "co2_factor"]].corr().iloc[::2, 1].to_frame("yearly_correlation"))
fig, ax = plt.subplots()
im = ax.hexbin(paired["cogeneration_kwh"] / 1_000_000, paired["co2_factor"] * 1000, gridsize=45, mincnt=1, bins="log")
ax.set(xlabel="Cogeneration energy (GWh per hourly interval)", ylabel="CO₂ factor (g CO₂/kWh)", title="Historical cogeneration production and total-mix carbon intensity")
fig.colorbar(im, ax=ax, label="log10(hour count)")
plt.tight_layout()
plt.show()

### Comparison with solar, onshore wind, and offshore wind

Compare aligned observations to assess co-movement. Report available pairs and unmatched hours for each series; leave missing hours unfilled. The correlation matrix uses only hours with all four production series observed. Associations can reflect shared seasonality and dispatch and do not establish incremental forecasting value.

In [ ]:
production = {"cogeneration": s}
comparison_rows = []
for name in ["sun", "wind_land", "wind_sea"]:
    other_files = sorted((ROOT / "data/raw" / name).glob("*.csv"))
    other = pd.concat([pd.read_csv(p) for p in other_files], ignore_index=True)
    for col in ["validfrom (UTC)", "validto (UTC)"]:
        other[col] = pd.to_datetime(other[col], utc=True, errors="raise")
    assert other["validfrom (UTC)"].is_unique
    assert (other["validto (UTC)"] - other["validfrom (UTC)"]).eq(pd.Timedelta(hours=1)).all()
    other = other.set_index("validfrom (UTC)").sort_index()
    production[name] = other[TARGET]
    pair = pd.concat([s.rename("cogeneration"), other[TARGET].rename(name)], axis=1).dropna()
    assert cogeneration_end.reindex(pair.index).eq(other.loc[pair.index, "validto (UTC)"]).all()
    comparison_rows.append({
        "series": name, "paired_hours": len(pair),
        "cogeneration_hours_without_other": len(s.dropna().index.difference(other[TARGET].dropna().index)),
        "other_hours_without_cogeneration": len(other[TARGET].dropna().index.difference(s.dropna().index)),
        "Pearson_correlation": pair.iloc[:, 0].corr(pair.iloc[:, 1]),
    })
display(pd.DataFrame(comparison_rows).set_index("series"))
production_complete = pd.DataFrame(production).dropna()
print(f"Complete observations across all four production series: {len(production_complete):,} hours")
production_corr = production_complete.corr()
display(production_corr)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(production_corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set(xticks=range(4), yticks=range(4), xticklabels=production_corr.columns, yticklabels=production_corr.index,
       title="Production correlations on common observed hours")
for row in range(4):
    for col in range(4):
        ax.text(col, row, f"{production_corr.iloc[row, col]:.2f}", ha="center", va="center")
fig.colorbar(im, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

## 8. Findings and next steps

In these exports, production is higher in winter and lower in summer in each observed year. Summer 2025 and 2026 levels are below summer 2024; this is a descriptive change, not an identified structural break or evidence of its cause. Year-specific profiles and coverage counts should accompany comparisons. Cogeneration has stronger daily and weekly persistence than the wind series, which motivates checking daily and weekly baselines rather than assuming added model value.

The missing period is 2026-08-22 07:00 through 2026-08-23 08:00 UTC inclusive (26 hours), overlapping the offshore-wind gap and extending one hour beyond it. A shared reporting issue is possible but unverified. Keep these hours missing.

Cogeneration is positively associated with total-mix carbon intensity and negatively associated with solar production, while full-sample linear associations with both wind series are weak. These relationships can reflect common seasonal or dispatch patterns. The nearly constant source emission factor makes source emissions largely redundant with energy; it is not the forecasting target.

The summary below is computed from the input files. Consult the coverage tables and seasonal charts before interpreting changes between years.

In [ ]:
display(Markdown(f"""
- Loaded **{len(files)} files**, **{len(df):,} rows**; UTC interval coverage **{start.min()} – {end.max()}** (end exclusive).
- Missing raw cells: **{df[raw_columns].isna().sum().sum():,}**; missing interior hours: **{len(missing_hours):,}**; duplicate-start rows: **{duplicate_times.sum():,}**; non-hour intervals: **{durations.ne(1).sum():,}**.
- Hourly cogeneration energy: **{s.min() / 1e6:.3f}–{s.max() / 1e6:.3f} GWh**, mean **{s.mean() / 1e6:.3f} GWh**; zero-output hours: **{zero.sum():,}**. Raw values are retained.
- Maximum absolute power–energy discrepancy: **{energy_error.abs().max():.3f} kWh**. Source-specific CO₂ factor rounds to **{factor.median():.3f} kg CO₂/kWh**, with **{factor.round(6).nunique()}** unique value(s) at six decimals; maximum emissions residual is **{emission_error.abs().max():.3f} kg CO₂**. Source emissions mostly repeat generation scaled by this factor and do not replace the total-mix target.
- IQR production flags: **{flags[TARGET].sum():,}**; large hourly changes: **{jump_flags.sum():,}** above **{jump_threshold / 1e6:.3f} GWh**. Flags are retained, not classified as errors.
- Autocorrelation at 24 / 168 hours: **{s.autocorr(24):.3f} / {s.autocorr(168):.3f}**. These lags alone do not establish daily or weekly recurrence.
- Aligned cogeneration–CO₂ Pearson correlation: **{correlations["all_hours"]:.3f}**. This does not establish causality or forecast skill. Compare with yearly correlations and the renewable-production matrix above.
- **2026 is partial**; compare matching calendar periods and check monthly coverage before interpreting annual production differences.

Next: assess interval energy as an optional additional input alongside renewable generation. Preserve UTC alignment and leave missing periods unfilled until a modeling policy is selected. Avoid redundant power, energy, and source-emission features. Do not infer renewable status or fuel mix from this export. Evaluate incremental value chronologically at 168-hour forecast origins, fit preprocessing on training data only, and compare persistence, daily, and weekly baselines. State that evaluation with future actual generation assumes known production and excludes generation forecast error; establish operational input availability separately.
"""))